In [1]:
# ============================================
# 패키지 설치 (Colab 첫 실행 시 1회만)
# ============================================
!pip -q install -U transformers datasets accelerate scikit-learn

print("✅ 패키지 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 17.4 MB/s eta 0:00:00
✅ 패키지 설치 완료


In [2]:
# ============================================
# Google Drive 연동
# ============================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================
# 작업자 설정 (여기만 수정하세요!)
# ============================================

# 1. 데이터셋 선택
DATASET = "AUDIT"  # "AUDIT", "ETC", "MDA" 중 하나 선택

# 2. 담당 Fold 선택
RUN_FOLDS = [1, 2]  # [1, 2] 또는 [3, 4] 선택

# ============================================
# 작업 분담표
# ============================================
# AUDIT 팀:
#   - 사람1: DATASET="AUDIT", RUN_FOLDS=[1,2]
#   - 사람2: DATASET="AUDIT", RUN_FOLDS=[3,4]
#
# ETC 팀:
#   - 사람3: DATASET="ETC", RUN_FOLDS=[1,2]
#   - 사람4: DATASET="ETC", RUN_FOLDS=[3,4]
#
# MDA 팀:
#   - 사람5: DATASET="MDA", RUN_FOLDS=[1,2]
#   - 사람6: DATASET="MDA", RUN_FOLDS=[3,4]
# ============================================

print("="*60)
print("🎯 작업 설정")
print("="*60)
print(f"데이터셋: {DATASET}")
print(f"담당 Fold: {RUN_FOLDS}")
print()
print(f"입력 파일: /content/drive/MyDrive/dataset_{DATASET}.csv")
print(f"출력 폴더: /content/drive/MyDrive/kobigbird_oof/")
print("="*60)


🎯 작업 설정
데이터셋: AUDIT
담당 Fold: [1, 2]

입력 파일: /content/drive/MyDrive/dataset_AUDIT.csv
출력 폴더: /content/drive/MyDrive/kobigbird_oof/


In [4]:
# ============================================
# 경로 설정
# ============================================
import os

CSV_PATH = f"/content/drive/MyDrive/dataset_{DATASET}.csv"
OUTPUT_DIR = "/content/drive/MyDrive/kobigbird_oof"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================
# 모델 및 하이퍼파라미터 설정
# ============================================
MODEL_NAME = "monologg/kobigbird-bert-base"
MAX_LEN = 2048  # BigBird는 2048 토큰까지 지원

N_FOLDS = 4
SEED = 42

# 학습 설정 (메모리 부족 시 TRAIN_BS 줄이거나 GRAD_ACCUM 늘리기)
EPOCHS = 1
LR = 2e-5
TRAIN_BS = 1         # GPU 배치 크기
EVAL_BS = 2
GRAD_ACCUM = 8       # Effective batch = TRAIN_BS * GRAD_ACCUM = 8
FP16 = True          # Mixed precision (T4/A100 권장)

print("📋 설정 정보")
print(f"  모델: {MODEL_NAME}")
print(f"  최대 길이: {MAX_LEN} 토큰")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {TRAIN_BS} x {GRAD_ACCUM} = {TRAIN_BS * GRAD_ACCUM}")
print(f"  Learning Rate: {LR}")
print(f"  FP16: {FP16}")


📋 설정 정보
  모델: monologg/kobigbird-bert-base
  최대 길이: 2048 토큰
  Epochs: 1
  Batch Size: 1 x 8 = 8
  Learning Rate: 2e-05
  FP16: True


In [5]:
# ============================================
# 라이브러리 임포트
# ============================================
import json
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import torch
from torch import nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

set_seed(SEED)
print("✅ 라이브러리 로드 완료")


✅ 라이브러리 로드 완료


In [6]:
# ============================================
# 데이터 로드
# ============================================
print("="*60)
print("📂 데이터 로드 중...")
print("="*60)

df = pd.read_csv(CSV_PATH)

# 필수 컬럼 확인
required_cols = ["corp_code", "target_year", "y", "text"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}\n현재 컬럼: {df.columns.tolist()}")

print(f"✅ 데이터 로드 완료")
print(f"   총 샘플: {len(df):,}개")
print(f"   컬럼: {df.columns.tolist()}")
print()

# 결측 처리
df["text"] = df["text"].fillna("")
df["y"] = df["y"].astype(int)

# 라벨 분포
print("📊 라벨 분포:")
print(df["y"].value_counts())
print()

# ============================================
# 4-Fold 분할 (기업 단위, StratifiedKFold)
# ============================================
print("="*60)
print("📦 4-Fold 분할 (기업 단위)")
print("="*60)

# 기업 단위 라벨: 한 번이라도 부도(y=1)이면 1
firm = (
    df.groupby("corp_code", as_index=False)
      .agg(firm_y=("y", "max"), n_rows=("y", "size"))
)

print(f"총 기업 수: {len(firm):,}개")
print(f"기업별 평균 샘플 수: {firm['n_rows'].mean():.1f}개")
print()

# StratifiedKFold로 기업을 4개 Fold로 분할
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

firm["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(firm["corp_code"], firm["firm_y"]), start=1):
    firm.loc[val_idx, "fold"] = fold_idx

# 기업별 Fold 매핑을 원본 데이터에 부여
fold_map = dict(zip(firm["corp_code"], firm["fold"]))
df["fold"] = df["corp_code"].map(fold_map).astype(int)

# ============================================
# 검증: Fold 간 기업 중복 없음
# ============================================
print("🔍 Fold 간 기업 중복 검증...")
overlap_found = False

for i in range(1, N_FOLDS+1):
    for j in range(i+1, N_FOLDS+1):
        firms_i = set(firm.loc[firm["fold"]==i, "corp_code"])
        firms_j = set(firm.loc[firm["fold"]==j, "corp_code"])
        inter = firms_i.intersection(firms_j)
        if len(inter) != 0:
            print(f"❌ Fold {i}와 {j}에서 기업 {len(inter)}개 중복!")
            overlap_found = True

if not overlap_found:
    print("✅ Fold 간 기업 중복 없음 (검증 통과)")
print()

# ============================================
# Fold별 분포 확인
# ============================================
print("📊 Fold별 분포 (기업 단위):")
print(firm.groupby("fold")["corp_code"].count().rename("기업 수"))
print()

print("📊 Fold별 라벨 분포 (기업 단위, ever-default):")
print(firm.groupby(["fold", "firm_y"]).size().unstack(fill_value=0))
print()

print("📊 Fold별 샘플 분포:")
print(df.groupby("fold")["y"].count().rename("샘플 수"))
print()

# Fold 매핑 저장
fold_map_file = os.path.join(OUTPUT_DIR, f"fold_map_{DATASET}_4fold_seed{SEED}.csv")
firm.to_csv(fold_map_file, index=False, encoding="utf-8-sig")
print(f"💾 Fold 매핑 저장: {fold_map_file}")
print("   (다른 팀원과 동일한 Fold 사용 시 이 파일 공유)")


📂 데이터 로드 중...
✅ 데이터 로드 완료
   총 샘플: 28,222개
   컬럼: ['corp_code', 'target_year', 'y', 'text']

📊 라벨 분포:
y
0    22174
1     6048
Name: count, dtype: int64

📦 4-Fold 분할 (기업 단위)
총 기업 수: 2,976개
기업별 평균 샘플 수: 9.5개

🔍 Fold 간 기업 중복 검증...
✅ Fold 간 기업 중복 없음 (검증 통과)

📊 Fold별 분포 (기업 단위):
fold
1    744
2    744
3    744
4    744
Name: 기업 수, dtype: int64

📊 Fold별 라벨 분포 (기업 단위, ever-default):
firm_y    0    1
fold            
1       568  176
2       568  176
3       569  175
4       569  175

📊 Fold별 샘플 분포:
fold
1    7071
2    7198
3    7245
4    6708
Name: 샘플 수, dtype: int64

💾 Fold 매핑 저장: /content/drive/MyDrive/kobigbird_oof/fold_map_AUDIT_4fold_seed42.csv
   (다른 팀원과 동일한 Fold 사용 시 이 파일 공유)


In [7]:
# ============================================
# Tokenizer 로드
# ============================================
print("="*60)
print("🔧 Tokenizer 설정")
print("="*60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):
    """배치 토크나이징 함수"""
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
    )

# DataCollator: 동적 패딩 (메모리 효율)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8  # 8의 배수로 패딩 (GPU 효율)
)

print(f"✅ Tokenizer 로드 완료")
print(f"   모델: {MODEL_NAME}")
print(f"   최대 길이: {MAX_LEN} 토큰")


🔧 Tokenizer 설정


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

✅ Tokenizer 로드 완료
   모델: monologg/kobigbird-bert-base
   최대 길이: 2048 토큰


In [8]:
# ============================================
# Class-weighted Trainer (불균형 데이터 처리)
# ============================================
class WeightedTrainer(Trainer):
    """클래스 가중치를 적용한 Trainer"""

    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k:v for k,v in inputs.items() if k!="labels"})
        logits = outputs.get("logits")

        # CrossEntropyLoss with class weights
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )
        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    """평가 메트릭: ROC-AUC"""
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    labels = np.array(labels)

    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = float("nan")

    return {"auc": auc}


print("✅ WeightedTrainer 정의 완료")


✅ WeightedTrainer 정의 완료


In [12]:
# ============================================
# Fold 학습 및 예측 함수
# ============================================
def train_and_predict_fold(fold):
    """
    단일 Fold 학습 및 OOF 예측

    Args:
        fold (int): 실행할 Fold 번호 (1, 2, 3, 4)

    Returns:
        dict: 결과 정보 (파일 경로, 메트릭 등)
    """
    print(f"\n{'='*60}")
    print(f"📦 Fold {fold}/{N_FOLDS} 시작")
    print(f"{'='*60}")

    # 1. Train/Val 분할
    train_df = df[df["fold"] != fold].copy()
    valid_df = df[df["fold"] == fold].copy()

    print(f"📊 데이터 분할:")
    print(f"   Train: {len(train_df):,}개 샘플 (Fold {fold} 제외)")
    print(f"   Val:   {len(valid_df):,}개 샘플 (Fold {fold})")
    print()

    # 2. 클래스 가중치
    y_train = train_df["y"].values
    n0 = (y_train == 0).sum()
    n1 = (y_train == 1).sum()
    w0 = (len(y_train) / (2.0 * n0)) if n0 > 0 else 1.0
    w1 = (len(y_train) / (2.0 * n1)) if n1 > 0 else 1.0
    class_weights = torch.tensor([w0, w1], dtype=torch.float)

    print(f"⚖️  클래스 가중치: [{w0:.3f}, {w1:.3f}]")
    print()

    # 3. Dataset 생성 및 토크나이징
    print("🔄 토크나이징 중...")
    train_ds = Dataset.from_pandas(
        train_df[["text", "y", "corp_code", "target_year", "fold"]],
        preserve_index=False
    ).rename_column("y", "labels").map(
        tokenize_batch, batched=True, remove_columns=["text"]
    )

    valid_ds = Dataset.from_pandas(
        valid_df[["text", "y", "corp_code", "target_year", "fold"]],
        preserve_index=False
    ).rename_column("y", "labels").map(
        tokenize_batch, batched=True, remove_columns=["text"]
    )
    print("✅ 토크나이징 완료\n")

    # 4. 모델 초기화
    print("🤖 모델 로딩...")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.gradient_checkpointing_enable()
    print("✅ 모델 로드 완료\n")

    # 5. Trainer 설정
    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, f"model_{DATASET}_fold{fold}"),
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        eval_strategy="epoch",              # ← evaluation_strategy → eval_strategy
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="auc",
        greater_is_better=True,
        fp16=FP16,
        logging_steps=50,
        report_to="none",
        seed=SEED,
    )


    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    # 6. 학습
    print(f"🚀 학습 시작 (Fold {fold})...\n")
    trainer.train()
    print(f"\n✅ Fold {fold} 학습 완료\n")

    # 7. 평가
    eval_metrics = trainer.evaluate()
    print(f"📊 Fold {fold} 평가 결과:")
    print(f"   AUC: {eval_metrics.get('eval_auc', 'N/A'):.4f}")
    print(f"   Loss: {eval_metrics.get('eval_loss', 'N/A'):.4f}\n")

    # 8. OOF 예측
    print(f"🔮 Fold {fold} OOF 예측 중...")
    preds = trainer.predict(valid_ds)
    logits = preds.predictions
    prob1 = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]

    # 9. 결과 저장
    out = valid_df[["corp_code", "target_year", "y", "fold"]].copy()
    out["pred_prob"] = prob1

    fold_file = os.path.join(OUTPUT_DIR, f"oof_pred_{DATASET}_fold{fold}_seed{SEED}.csv")
    out.to_csv(fold_file, index=False, encoding="utf-8-sig")

    metrics_file = os.path.join(OUTPUT_DIR, f"metrics_{DATASET}_fold{fold}_seed{SEED}.json")
    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump({"dataset": DATASET, "fold": fold, "metrics": eval_metrics},
                  f, ensure_ascii=False, indent=2)

    # 10. 메모리 정리
    del model, trainer
    torch.cuda.empty_cache()

    # 결과 반환
    result = {
        "fold": fold,
        "dataset": DATASET,
        "n_samples": len(out),
        "auc": eval_metrics.get('eval_auc', None),
        "loss": eval_metrics.get('eval_loss', None),
        "fold_file": fold_file,
        "metrics_file": metrics_file,
    }

    print("="*60)
    print(f"🎉 Fold {fold} 완료!")
    print("="*60)
    print(f"📄 파일: {fold_file}")
    print(f"📊 샘플: {len(out):,}개")
    print(f"📈 AUC: {result['auc']:.4f}")
    print("="*60)

    return result


print("✅ 함수 정의 완료")
print()
print("💡 사용법:")
print("   train_and_predict_fold(1)  # Fold 1 실행")
print("   train_and_predict_fold(2)  # Fold 2 실행")
print("   train_and_predict_fold(3)  # Fold 3 실행")
print("   train_and_predict_fold(4)  # Fold 4 실행")

✅ 함수 정의 완료

💡 사용법:
   train_and_predict_fold(1)  # Fold 1 실행
   train_and_predict_fold(2)  # Fold 2 실행
   train_and_predict_fold(3)  # Fold 3 실행
   train_and_predict_fold(4)  # Fold 4 실행


In [ ]:
# Fold 1 실행
result_fold1 = train_and_predict_fold(1)

# Fold 2 실행
# result_fold2 = train_and_predict_fold(2)

# Fold 3 실행
# result_fold3 = train_and_predict_fold(3)

# Fold 4 실행
# result_fold4 = train_and_predict_fold(4)


📦 Fold 1/4 시작
📊 데이터 분할:
   Train: 21,151개 샘플 (Fold 1 제외)
   Val:   7,071개 샘플 (Fold 1)

⚖️  클래스 가중치: [0.638, 2.318]

🔄 토크나이징 중...


Map:   0%|          | 0/21151 [00:00<?, ? examples/s]

Map:   0%|          | 0/7071 [00:00<?, ? examples/s]

Some weights of BigBirdForSequenceClassification were not initialized from the model checkpoint at monologg/kobigbird-bert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ 토크나이징 완료

🤖 모델 로딩...
✅ 모델 로드 완료



/tmp/ipython-input-683906804.py:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


🚀 학습 시작 (Fold 1)...



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Input ids are automatically padded from 1016 to 1024 to be a multiple of `config.block_size`: 64
Attention type 'block_sparse' is not possible if sequence_length: 24 <= num global tokens: 2 * config.block_size + min. num sliding tokens: 3 * config.block_size + config.num_random_blocks * config.block_size + additional buffer: config.num_random_blocks * config.block_size = 704 with config.block_size = 64, config.num_random_blocks = 3. Changing attention type to 'original_full'...


In [ ]:
# Fold 1 실행
# result_fold1 = train_and_predict_fold(1)

# Fold 2 실행
result_fold2 = train_and_predict_fold(2)

# Fold 3 실행
# result_fold3 = train_and_predict_fold(3)

# Fold 4 실행
# result_fold4 = train_and_predict_fold(4)

In [ ]:
# ============================================
# 결과 확인
# ============================================
print("="*60)
print("📋 결과 확인")
print("="*60)
print()

# 부분 통합 파일 읽기
tag = "".join([str(x) for x in RUN_FOLDS])
partial_file = os.path.join(OUTPUT_DIR, f"oof_pred_{DATASET}_folds{tag}_seed{SEED}.csv")

result = pd.read_csv(partial_file)

print(f"📄 파일: {partial_file}")
print(f"📊 샘플 수: {len(result):,}개")
print()

print("📋 샘플 데이터 (상위 10개):")
print(result.head(10))
print()

print("📊 예측 확률 통계:")
print(result["pred_prob"].describe())
print()

print("📊 라벨별 평균 예측 확률:")
for label in sorted(result["y"].unique()):
    label_name = "부도" if label == 1 else "정상"
    avg_prob = result[result["y"]==label]["pred_prob"].mean()
    std_prob = result[result["y"]==label]["pred_prob"].std()
    count = (result["y"]==label).sum()
    print(f"   {label_name} (y={label}): {avg_prob:.4f} ± {std_prob:.4f} (n={count:,})")
print()

print("✅ 작업 완료!")
print()
print("📥 산출물:")
print(f"   - Fold별 CSV: {OUTPUT_DIR}/oof_pred_{DATASET}_fold*.csv")
print(f"   - 부분 통합: {partial_file}")
